# Further Experiment 5 — Pretraining ablation

Is DistilBERT's edge the **transformer architecture** or the **masked-LM pretraining**? Two probes, all with the identical fine-tuning recipe (40k, 1 epoch, lr 2e-5):

1. the **same DistilBERT architecture with random weights** (no pretraining), and
2. progressively **smaller but pretrained** BERTs (Tiny / Mini / Small).

**What we noticed:** pretraining is essentially the *whole* advantage. The random-init transformer (0.763) is no better than a plain LSTM and *loses* to TF-IDF; pretraining adds ~+6 points. And a tiny *pretrained* BERT beats the big *from-scratch* one — pretraining is worth ~6x the parameters — though there is a capacity floor (BERT-Tiny is too small).

## Setup

In [ ]:
!pip install -q datasets transformers accelerate scikit-learn

In [ ]:
import re, time, random, numpy as np, pandas as pd, torch
from datasets import load_dataset, concatenate_datasets, Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
                          BertForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # strip URLs
    text = re.sub(r'@\w+', '@user', text)                  # @mentions -> @user
    text = re.sub(r'\s+', ' ', text).strip()               # normalize whitespace
    return text

## Same 40k/10k split

In [ ]:
ds = load_dataset('contemmcm/sentiment140')['complete']
neg = ds.filter(lambda x: x['label'] == 0).shuffle(seed=SEED).select(range(25_000))
pos = ds.filter(lambda x: x['label'] == 1).shuffle(seed=SEED).select(range(25_000))
data = concatenate_datasets([neg, pos]).shuffle(seed=SEED)
X = [clean_tweet(t) for t in data['text']]; y = list(data['label'])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

## One fine-tuning helper, used for every model

We reuse the standard bert-base-uncased tokenizer for the small BERTs (their vocab is the same) because some of their repos ship a broken tokenizer file.

In [ ]:
TOK = AutoTokenizer.from_pretrained('bert-base-uncased')
def make_ds(texts, labels):
    def t(b): return TOK(b['text'], truncation=True, max_length=64)
    return HFDataset.from_pandas(pd.DataFrame({'text': texts, 'label': labels})).map(t, batched=True)
tr_ds, te_ds = make_ds(X_tr, y_tr), make_ds(X_te, y_te)

def finetune(model, tag):
    args = TrainingArguments(output_dir=f'./_{tag}', num_train_epochs=1,
        per_device_train_batch_size=32, per_device_eval_batch_size=64, learning_rate=2e-5,
        report_to='none', fp16=torch.cuda.is_available(), logging_strategy='no',
        save_strategy='no', seed=SEED)
    def m(ep):
        l, yy = ep; pr = np.argmax(l, 1)
        return {'accuracy': accuracy_score(yy, pr), 'f1': f1_score(yy, pr)}
    tr = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=te_ds,
                 data_collator=DataCollatorWithPadding(TOK), compute_metrics=m)
    tr.train(); r = tr.evaluate()
    n = sum(p.numel() for p in model.parameters())
    print(f'{tag:34s} params={n/1e6:5.1f}M  acc={r["eval_accuracy"]:.4f}')
    return r['eval_accuracy']

## Probe 1 — DistilBERT architecture, RANDOM init (no pretraining)

`from_config` gives the exact architecture with random weights.

In [ ]:
cfg = AutoConfig.from_pretrained('distilbert-base-uncased', num_labels=2)
torch.manual_seed(SEED)
scratch = AutoModelForSequenceClassification.from_config(cfg)
finetune(scratch, 'DistilBERT from scratch (no pretrain)')

## Probe 2 — smaller PRETRAINED BERTs

In [ ]:
for repo, tag in [('prajjwal1/bert-tiny',  'BERT-Tiny (pretrained)'),
                  ('prajjwal1/bert-mini',  'BERT-Mini (pretrained)'),
                  ('prajjwal1/bert-small', 'BERT-Small (pretrained)')]:
    torch.manual_seed(SEED)
    model = BertForSequenceClassification.from_pretrained(repo, num_labels=2)
    finetune(model, tag)

## What we found

| Model (40k, identical recipe) | Params | Pretrained? | Accuracy |
|---|---|---|---|
| BERT-Tiny | 4.4M | yes | 0.707 |
| BERT-Mini | 11.2M | yes | 0.776 |
| BERT-Small | 28.8M | yes | 0.804 |
| DistilBERT | 66M | yes | **0.825** |
| DistilBERT, from scratch | 66M | **no** | 0.763 |
| TF-IDF + LogReg (reference) | – | – | 0.796 |

- **Pretraining is ~the whole transformer advantage: +6 points** (0.763 -> 0.825). Without it the 66M transformer is no better than a plain LSTM and loses to TF-IDF.
- **A 29M *pretrained* BERT (0.804) beats the 66M *from-scratch* one (0.763)** — pretraining is worth ~6x the parameters — but there's a capacity floor: BERT-Tiny (4.4M) still loses to the big from-scratch model.
- Combined with Experiment 3 (pretrained-but-not-fine-tuned ≈ chance), the lesson is you need **both** pretraining and fine-tuning.